# IOAI — 2024 Summer National Machine Translation (Colab 자동 설정판)

아래 **설정 셀을 먼저 실행**하면 공개 데이터 소스에서 데이터를 받아 이 폴더에 `train.csv`/`test.csv` 등으로 준비합니다. 이후 셀이 그대로 학습/예측하고, 만들어진 제출 파일을 내려받아 연습 사이트 **Submissions** 탭에 올리면 채점됩니다.

> 런타임 메뉴 → **런타임 유형 변경 → GPU** (필요 시).

In [ ]:
# === 데이터 자동 준비 (가장 먼저 실행) ===
!pip -q install datasets
print('opus_books fr-hu 코퍼스는 노트북 첫 셀이 datasets 로 자동 다운로드합니다.')
import os; print('작업 폴더:', os.getcwd()); print('내용:', sorted(os.listdir('.')))

# Gépi Fordítás (Machine Translation) — FR→HU 모범답안

> **HAIO 2024 — Summer National Finals (NLP) · 50 pts**

A **sequence-to-sequence** (GRU encoder → GRU decoder) reference for French→Hungarian translation, scored by held-out **corpus BLEU**.

**Data note:** the organisers' original parallel data (`nyelvek.zip`) and the provided pretrained encoder/decoder weights were deleted from Google Drive. This reference reconstructs the *same task* on the public **`Helsinki-NLP/opus_books` (fr-hu)** corpus, restricted to short sentences (2–10 words), with the **same deterministic split** (every 10th pair held out) as the grader's `val_refs.csv`. It trains the model from scratch (the original loaded provided weights) and writes `submission.csv` (`id,hu`).

In [ ]:
!pip -q install datasets

In [ ]:
import numpy as np, pandas as pd, torch, torch.nn as nn, random
torch.manual_seed(0); random.seed(0); np.random.seed(0)
device='cuda' if torch.cuda.is_available() else 'cpu'
from datasets import load_dataset
ds=load_dataset('Helsinki-NLP/opus_books','fr-hu',split='train')
def norm(s):
    return s.strip().lower()
pairs=[]
for ex in ds:
    fr=norm(ex['translation']['fr']); hu=norm(ex['translation']['hu'])
    if not fr or not hu: continue
    nf,nh=fr.split(),hu.split()
    if not (2<=len(nf)<=10 and 2<=len(nh)<=10): continue
    pairs.append((fr,hu))
# ugyanaz a determinisztikus split, mint a val_refs.csv-nél: minden 10. pár held-out
train=[p for i,p in enumerate(pairs) if i%10!=0]
val  =[p for i,p in enumerate(pairs) if i%10==0]
print('pairs',len(pairs),'train',len(train),'val',len(val))

In [ ]:
from collections import Counter
PAD,SOS,EOS,UNK=0,1,2,3
def build_vocab(sents,maxv=8000):
    c=Counter(w for s in sents for w in s.split())
    itos=['<pad>','<sos>','<eos>','<unk>']+[w for w,_ in c.most_common(maxv)]
    return {w:i for i,w in enumerate(itos)}, itos
src_stoi,_=build_vocab([f for f,_ in train])
tgt_stoi,tgt_itos=build_vocab([h for _,h in train])
def enc(s,stoi): return [stoi.get(w,UNK) for w in s.split()]
def encode_pair(f,h):
    return enc(f,src_stoi), [SOS]+enc(h,tgt_stoi)+[EOS]
print('src vocab',len(src_stoi),'tgt vocab',len(tgt_stoi))

In [ ]:
from torch.nn.utils.rnn import pad_sequence
from torch.utils.data import DataLoader, Dataset
class PairDS(Dataset):
    def __init__(self,data): self.data=[encode_pair(f,h) for f,h in data]
    def __len__(self): return len(self.data)
    def __getitem__(self,i): return self.data[i]
def collate(b):
    src=[torch.tensor(x[0]) for x in b]; tgt=[torch.tensor(x[1]) for x in b]
    return (pad_sequence(src,batch_first=True,padding_value=PAD),
            pad_sequence(tgt,batch_first=True,padding_value=PAD))
class Seq2Seq(nn.Module):
    def __init__(self,vs,vt,emb=256,hid=512):
        super().__init__()
        self.es=nn.Embedding(vs,emb,padding_idx=PAD); self.et=nn.Embedding(vt,emb,padding_idx=PAD)
        self.enc=nn.GRU(emb,hid,batch_first=True)
        self.dec=nn.GRU(emb,hid,batch_first=True)
        self.out=nn.Linear(hid,vt)
    def forward(self,src,tgt):
        _,h=self.enc(self.es(src))
        o,_=self.dec(self.et(tgt[:,:-1]),h)
        return self.out(o)
    @torch.no_grad()
    def greedy(self,src,maxlen=12):
        _,h=self.enc(self.es(src)); B=src.size(0)
        tok=torch.full((B,1),SOS,device=src.device); outs=[]
        for _ in range(maxlen):
            o,h=self.dec(self.et(tok),h); nx=self.out(o[:,-1]).argmax(-1,keepdim=True)
            outs.append(nx); tok=nx
        return torch.cat(outs,1)
m=Seq2Seq(len(src_stoi),len(tgt_stoi)).to(device)
print(sum(p.numel() for p in m.parameters())/1e6,'M params')

In [ ]:
opt=torch.optim.Adam(m.parameters(),1e-3)
crit=nn.CrossEntropyLoss(ignore_index=PAD)
dl=DataLoader(PairDS(train),batch_size=128,shuffle=True,collate_fn=collate)
EPOCHS=12   # a referencia tovább tanul, mint a baseline (8)
for ep in range(EPOCHS):
    m.train(); tot=0
    for src,tgt in dl:
        src,tgt=src.to(device),tgt.to(device)
        logits=m(src,tgt)
        loss=crit(logits.reshape(-1,logits.size(-1)),tgt[:,1:].reshape(-1))
        opt.zero_grad(); loss.backward(); opt.step(); tot+=loss.item()
    print(f'epoch {ep+1}/{EPOCHS} loss {tot/len(dl):.3f}')

In [ ]:
# decode held-out val → Hungarian → submission.csv
m.eval(); preds=[]
vdl=DataLoader(PairDS(val),batch_size=256,shuffle=False,collate_fn=collate)
for src,_ in vdl:
    out=m.greedy(src.to(device)).cpu().tolist()
    for seq in out:
        words=[]
        for t in seq:
            if t==EOS: break
            if t>=4: words.append(tgt_itos[t])
        preds.append(' '.join(words))
refs=[h for _,h in val]
pd.DataFrame({'id':range(len(preds)),'hu':preds}).to_csv('submission.csv',index=False)
print('wrote submission.csv',len(preds))
print('sample:',preds[1],'||ref:',refs[1])

In [ ]:
# self-check BLEU (same formula as the grader)
import math
from collections import Counter
def toks(s): return str(s).split()
def ng(t,n): return Counter(tuple(t[i:i+n]) for i in range(len(t)-n+1)) if len(t)>=n else Counter()
def cbleu(P,R,N=4):
    clip=[0]*N; tot=[0]*N; pl=rl=0
    for p,r in zip(P,R):
        pt,rt=toks(p),toks(r); pl+=len(pt); rl+=len(rt)
        for n in range(1,N+1):
            pc,rc=ng(pt,n),ng(rt,n); tot[n-1]+=sum(pc.values())
            clip[n-1]+=sum(min(c,rc.get(g,0)) for g,c in pc.items())
    pr=[(clip[n]+1e-9)/(tot[n]+1e-9) for n in range(N)]
    bp=1.0 if pl>rl else math.exp(1-rl/max(pl,1e-9))
    return bp*math.exp(sum(math.log(p) for p in pr)/N)
print('held-out BLEU:',round(cbleu(preds,refs),4))

## 제출 파일 모으기
아래 셀을 실행하면 제출 파일이 **최상위(`/content`)로 복사**되어 왼쪽 파일 탐색기에 바로 보입니다.
그 파일을 내려받아 연습 사이트 **Submissions** 탭에 올리면 채점됩니다.

In [ ]:
# === 제출 파일을 /content 로 모으기 (마지막에 실행) ===
import os, glob, shutil
TARGETS = ['submission.csv']
OUT = "/content" if os.path.isdir("/content") else os.getcwd()
found = []
for name in TARGETS:
    hits = [name] if os.path.exists(name) else glob.glob(f"**/{name}", recursive=True)
    if not hits:
        print("아직 없음(해당 셀을 먼저 실행하세요):", name); continue
    dst = os.path.join(OUT, os.path.basename(hits[0]))
    if os.path.abspath(hits[0]) != os.path.abspath(dst):
        shutil.copy2(hits[0], dst)
    found.append(dst)
print("제출 파일 저장 위치(파일 탐색기 최상위):", found)